In [79]:
import pandas as pd
import numpy as np

fact = pd.read_csv("fact_school_fees.csv")
schools = pd.read_csv("../../data/data_correction/dim_school.csv")
standards = pd.read_csv("../../data/dashboard_local/data/dim_standard.csv")

fact.columns = fact.columns.str.strip()

In [80]:
year_median = (
    fact.groupby(['school_key','academic_year'])['approved_fee']
    .median()
    .reset_index()
    .rename(columns={'approved_fee':'year_median'})
)

fact = fact.merge(year_median,
                  on=['school_key','academic_year'],
                  how='left')

In [81]:
std_median = (
    fact.groupby(['school_key','standard_id'])['approved_fee']
    .median()
    .reset_index()
    .rename(columns={'approved_fee':'standard_median'})
)

fact = fact.merge(std_median,
                  on=['school_key','standard_id'],
                  how='left')

In [82]:
fact['ratio_year'] = fact['approved_fee'] / fact['year_median']
fact['ratio_standard'] = fact['approved_fee'] / fact['standard_median']

In [83]:
def adjust_power10(value, ref):
    if pd.isna(ref) or ref == 0:
        return value

    adj = value

    while adj > 2 * ref:
        adj /= 10

    while adj < 0.5 * ref:
        adj *= 10

    return round(adj,2)

In [84]:
def should_correct(row):

    fee = row['approved_fee']
    yr = row['year_median']
    std = row['standard_median']

    if pd.isna(yr) or yr == 0:
        return False

    # abnormal vs peers
    peer_outlier = (fee > 5 * yr) or (fee < 0.2 * yr)

    # abnormal vs own history
    hist_outlier = False
    if not pd.isna(std) and std > 0:
        hist_outlier = (fee > 5 * std) or (fee < 0.2 * std)

    # only correct if BOTH conditions true
    return peer_outlier and hist_outlier

In [85]:
fact['correction_flag'] = fact.apply(should_correct, axis=1)

fact['approved_fee_corrected'] = fact.apply(
    lambda r: adjust_power10(r['approved_fee'], r['year_median'])
    if r['correction_flag']
    else r['approved_fee'],
    axis=1
)

In [86]:
fact_final = fact.copy()

fact_final['approved_fee'] = fact_final['approved_fee_corrected']

fact_final = fact_final[
    ['fee_key','school_key','standard_id','academic_year','approved_fee','inserted_at']
]

fact_final.to_csv("fact_school_fees_corrected_final.csv", index=False)

In [87]:
audit = fact[fact['correction_flag']==True].copy()

audit = audit.merge(
    schools[['school_key','school_name']],
    on='school_key', how='left'
)

audit = audit.merge(
    standards[['standard_id','standard_name']],
    on='standard_id', how='left'
)

audit_table = audit[
    ['school_name','standard_name','academic_year',
     'approved_fee','approved_fee_corrected']
]

audit_table.rename(columns={
    'approved_fee':'previous_fee',
    'approved_fee_corrected':'corrected_fee'
}, inplace=True)

audit_table.to_csv("../../data/data_correction/fee_correction_audit_report.csv", index=False)

C:\Users\abhis\AppData\Local\Temp\ipykernel_23276\2229420845.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  audit_table.rename(columns={


In [88]:
print("Total:", len(fact))
print("Corrected:", fact['correction_flag'].sum())
print("Percent corrected:", round(fact['correction_flag'].mean()*100,2))

Total: 1238616
Corrected: 578
Percent corrected: 0.05
